# Laboratorio 01 · Python scientifico per l’ottimizzazione
**Ottimizzazione: Modelli e Metodi** · Laurea triennale in Statistica, Università degli Studi di Padova

**Anthony Palmieri** · Dottorando in Computational Mathematics

Questo è il primo laboratorio operativo del percorso, dopo la presentazione introduttiva.
Ripassiamo Python, organizziamo dati con NumPy e pandas, incontriamo SciPy e risolviamo
due piccoli problemi con **SciPy/HiGHS e Gurobi**.

<a id="preparazione"></a>
## 0. Preparazione

In Colab apri il notebook, salva una tua copia e usa **Shift+Invio** per eseguire una cella.
Le celle Markdown contengono spiegazioni; le celle di codice contengono istruzioni Python.
Esegui dall’alto verso il basso: le variabili appartengono al **runtime**, cioè alla sessione
di calcolo. Salvare il notebook non conserva automaticamente memoria, pacchetti installati
e file temporanei del runtime. Quando la sessione viene ricreata, riesegui la preparazione.

Usiamo una **CPU standard**. Non servono GPU, abbonamenti, chiavi personali o accesso ai dati
di Drive. La connessione serve per Colab e per l’eventuale installazione del pacchetto.

`%pip` installa un pacchetto nell’ambiente del notebook. Installiamo soltanto `gurobipy`,
senza aggiornare indiscriminatamente le librerie preinstallate.

In [1]:
%pip -q install gurobipy


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Import e controllo dell’ambiente

`import numpy as np` importa una libreria e le assegna il nome breve `np`.
`from ... import ...` importa oggetti specifici. Il punto accede a una funzione o a
un’informazione della libreria: per esempio `np.__version__` è la sua versione.
`print(...)` mostra testo e valori; `display(...)` visualizza le tabelle nel notebook.
La cella seguente è preparazione tecnica: basta eseguirla una volta.
`hasattr` controlla che un attributo esista; `if not` entra nel blocco se manca.
In quel caso `raise ImportError(...)` segnala quale dipendenza occorre installare.

`milp` è disponibile da SciPy 1.9. Se il controllo fallisce in un ambiente locale vecchio,
installa **solo** una versione adatta di SciPy (`%pip install "scipy>=1.9"`),
riavvia il runtime e ricomincia dall’inizio.

In [2]:
import numpy as np
import pandas as pd
import scipy
import scipy.optimize
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

if not hasattr(scipy.optimize, "milp"):
    raise ImportError("Serve scipy.optimize.milp: installare SciPy >= 1.9 e riavviare.")
from scipy.optimize import milp, Bounds, LinearConstraint

print("NumPy:", np.__version__, "| pandas:", pd.__version__)
print("SciPy:", scipy.__version__, "| Gurobi:", gp.gurobi.version())

NumPy: 2.3.5 | pandas: 2.2.3
SciPy: 1.17.0 | Gurobi: (13, 0, 3)


<a id="python"></a>
## 1. Python: poche costruzioni da riutilizzare

### Variabili, tipi e operazioni

`=` assegna un valore a un nome. I tipi che incontreremo sono interi (`int`), numeri
decimali (`float`), stringhe (`str`) e booleani (`bool`, cioè `True`/`False`).
`None` rappresenta l’assenza di un valore: sarà utile quando un solver non restituisce
una soluzione. `type(...)` mostra il tipo.
`+`, `*` e `/` eseguono operazioni aritmetiche; `==`, `>` e `>=` confrontano valori
e restituiscono booleani. `and` richiede che entrambe le condizioni siano vere.

In [3]:
numero = 3
quota = 0.5
testo = "Python"
pronto = True
valore_assente = None
print(numero, type(numero), quota, type(quota))
print(testo, type(testo), pronto, type(pronto))
print(numero + 2, numero * 2, numero / 2)
print(numero == 3, numero >= 4)  # == confronta: non assegna
print(pronto and numero > 0)    # and richiede entrambe le condizioni

3 <class 'int'> 0.5 <class 'float'>
Python <class 'str'> True <class 'bool'>
5 6 1.5
True False
True


### I dati delle attività

Una **lista** contiene elementi ordinati, tra parentesi quadre. Le quattro liste qui sotto
sono allineate: la posizione 0 descrive A, la posizione 1 descrive B, e così via.
Questa è l’unica cella che definisce i dati base; array, tabelle e coefficienti saranno
ricavati da qui.

In [4]:
# DATI BASE: modificare qui, poi rieseguire dall'inizio.
id_attivita = ["A", "B", "C", "D", "E"]
nomi_attivita = ["Torneo sportivo", "Laboratorio foto", "Rassegna cinema",
                 "Scambio libri", "Concerto"]
costi_lista = [400, 300, 500, 200, 600]  # euro
benefici_lista = [8, 6, 9, 5, 11]       # punti
budget = 1000                          # euro

Gli indici partono da **zero**. `lista[inizio:fine]` seleziona dall’indice iniziale
fino a quello finale **escluso**; `len(lista)` restituisce la lunghezza.
Un indice negativo conta dalla fine: `-1` è l’ultimo elemento.

In [5]:
print("Prima attività:", id_attivita[0], nomi_attivita[0], costi_lista[0])
print("Ultima attività:", id_attivita[-1])
print("Posizioni 1 e 2:", id_attivita[1:3])
print("Numero di attività:", len(id_attivita))

Prima attività: A Torneo sportivo 400
Ultima attività: E
Posizioni 1 e 2: ['B', 'C']
Numero di attività: 5


### Dizionari, condizioni e cicli

Un **dizionario** associa chiavi a valori con la sintassi `{chiave: valore}`.
Si accede tramite la chiave, non tramite una posizione.
`.keys()` e `.values()` mostrano rispettivamente chiavi e valori.
Il punto introduce anche un **metodo**, cioè un’operazione offerta dall’oggetto.
`if` esegue un blocco quando la condizione è vera; `else` gestisce l’altro caso.
Il blocco inizia dopo `:` e si riconosce dai **rientri**.

In [6]:
prima_attivita = {"id": id_attivita[0], "costo": costi_lista[0]}
print(prima_attivita["id"], prima_attivita["costo"])
print(prima_attivita.keys(), prima_attivita.values())
if prima_attivita["costo"] <= budget:
    print("Questa singola attività rientra nel budget.")
else:
    print("Questa singola attività supera il budget.")

A 400
dict_keys(['id', 'costo']) dict_values(['A', 400])
Questa singola attività rientra nel budget.


`for` ripete un blocco. `range(n)` produce gli interi da 0 a `n-1`.
Un dizionario vuoto si scrive `{}`; `dizionario[chiave] = valore` aggiunge un’associazione.

In [7]:
costi_per_id = {}
for posizione in range(len(id_attivita)):
    identificativo = id_attivita[posizione]
    costi_per_id[identificativo] = costi_lista[posizione]
print(costi_per_id)
print("Costo di C:", costi_per_id["C"])

{'A': 400, 'B': 300, 'C': 500, 'D': 200, 'E': 600}
Costo di C: 500


`enumerate(lista)` fornisce coppie **(posizione, elemento)**.
`zip(lista1, lista2)` fornisce coppie di elementi alla stessa posizione;
si ferma alla lista più corta, quindi qui le liste devono avere la stessa lunghezza.
Le due variabili dopo `for` ricevono i due elementi della coppia.

In [8]:
for posizione, identificativo in enumerate(id_attivita):
    print(posizione, identificativo)

benefici_per_id = {}
for identificativo, beneficio in zip(id_attivita, benefici_lista):
    benefici_per_id[identificativo] = beneficio
print(benefici_per_id)

0 A
1 B
2 C
3 D
4 E
{'A': 8, 'B': 6, 'C': 9, 'D': 5, 'E': 11}


### Da un ciclo a una comprensione

`.append(...)` aggiunge un elemento in fondo a una lista.
Una **comprensione** scrive la stessa costruzione in una riga, nell’ordine
“valore da inserire, ciclo, eventuale filtro”.

In [9]:
economiche = []
for identificativo in id_attivita:
    if costi_per_id[identificativo] < 400:
        economiche.append(identificativo)
print("Con il ciclo:", economiche)

economiche_compatte = [j for j in id_attivita if costi_per_id[j] < 400]
costi_compatti = {j: costo for j, costo in zip(id_attivita, costi_lista)}
print("Con la comprensione:", economiche_compatte)
print(costi_compatti)  # equivale al dizionario costruito con il ciclo

Con il ciclo: ['B', 'D']
Con la comprensione: ['B', 'D']
{'A': 400, 'B': 300, 'C': 500, 'D': 200, 'E': 600}


**Micro-esercizio 1** Crea con un ciclo una lista `risposta_1` contenente
gli ID delle attività di costo **minore o uguale a 400 euro**, nell’ordine originario.
Usa `costi_per_id`; non modificare i dati. [Soluzione](#soluzione-1).

In [ ]:
# Tocca a te: crea risposta_1 e stampala.

### Funzioni

`sum`, `min` e `max` calcolano somma, minimo e massimo di una sequenza numerica.
Partiamo da una scelta ammissibile: A, B e D. `in` verifica l’appartenenza a una lista.

In [11]:
scelta_esempio = ["A", "B", "D"]
totale_esempio = 0
for identificativo in scelta_esempio:
    totale_esempio = totale_esempio + costi_per_id[identificativo]
print("Costo della scelta:", totale_esempio)
print("Somma di tutti i costi:", sum(costi_lista))
print("Costo minimo:", min(costi_lista), "Beneficio massimo:", max(benefici_lista))
print("B è selezionata?", "B" in scelta_esempio)

Costo della scelta: 900
Somma di tutti i costi: 2000
Costo minimo: 200 Beneficio massimo: 11
B è selezionata? True


`def` definisce una funzione. I **parametri** sono gli ingressi e `return` restituisce
il risultato. Qui incapsuliamo il procedimento appena eseguito, senza dipendere
da variabili nascoste dentro la funzione.

In [12]:
def costo_scelta(identificativi_scelti, dizionario_costi):
    totale = 0
    for identificativo in identificativi_scelti:
        totale = totale + dizionario_costi[identificativo]
    return totale

print(costo_scelta(scelta_esempio, costi_per_id))

900


**Come leggere un errore.** Se scrivessi `costi_per_id["Z"]`, otterresti
`KeyError: 'Z'`: la chiave Z non esiste. `costi_lista[5]` darebbe `IndexError`,
perché gli indici validi sono 0–4. Leggi il tipo di errore, il messaggio finale e
la riga indicata.

Più avanti useremo `x is not None` per controllare che un valore esista,
`not` per negare una condizione e `assert condizione` per fermare l’esecuzione
se una verifica necessaria fallisce.

<a id="numpy"></a>
## 2. NumPy: vettori e matrici

### Array e operazioni numeriche

Un `ndarray` è un array di elementi di uno stesso tipo. `np.array(...)` converte
le nostre liste. `shape` descrive la forma, `ndim` il numero di assi, `dtype` il
tipo degli elementi. Scriviamo `dtype=float` per usare numeri decimali.

In [13]:
costi = np.array(costi_lista, dtype=float)
benefici = np.array(benefici_lista, dtype=float)
identificativi = np.array(id_attivita)
print(costi)
print("shape:", costi.shape, "ndim:", costi.ndim, "dtype:", costi.dtype)
print("Lista per 2:", costi_lista[:2] * 2)  # ripete la lista
print("Array per 2:", costi[:2] * 2)        # moltiplica gli elementi

[400. 300. 500. 200. 600.]
shape: (5,) ndim: 1 dtype: float64
Lista per 2: [400, 300, 400, 300]
Array per 2: [800. 600.]


Un confronto tra array e scalare produce una **maschera booleana**.
Usando la maschera come indice manteniamo solo gli elementi in corrispondenza di `True`.
Gli indici e gli slicing di base funzionano come per le liste.
Il **broadcasting** estende uno scalare a tutti gli elementi: `costi * 0.9` applica
lo stesso fattore a ogni costo.

In [14]:
maschera = costi <= 400
print("Maschera:", maschera)
print("ID:", identificativi[maschera], "Costi:", costi[maschera])
print("Primi due costi:", costi[:2])
print("Costi scontati del 10%:", costi * 0.9)  # costi originali invariati

Maschera: [ True  True False  True False]
ID: ['A' 'B' 'D'] Costi: [400. 300. 200.]
Primi due costi: [400. 300.]
Costi scontati del 10%: [360. 270. 450. 180. 540.]


### Decisioni, `*` e `@`

Rappresentiamo la scelta A, B, D con un vettore: una componente vale 1 se l’ID è
in `scelta_esempio`, 0 altrimenti. L’ordine è sempre **A, B, C, D, E**.

- `costi * x` calcola i contributi **elemento per elemento**.
- `costi @ x` è il **prodotto scalare**: $\sum_j a_jx_j$.

La conversione `dtype=int` trasforma i booleani `True`/`False` in 1/0.

In [15]:
x_esempio = np.array([j in scelta_esempio for j in id_attivita], dtype=int)
print("Decisioni:", x_esempio)
print("Contributi al costo:", costi * x_esempio)
print("Costo:", costi @ x_esempio)
print("Beneficio:", benefici @ x_esempio)

Decisioni: [1 1 0 1 0]
Contributi al costo: [400. 300.   0. 200.   0.]
Costo: 900.0
Beneficio: 19.0


**Micro-esercizio 2** Costruisci il vettore binario per scegliere **B e D**,
nell’ordine A–E. Calcola costo e beneficio con `@` e confronta il costo con `budget`.
Puoi partire da una lista di cinque zeri e uno. [Soluzione](#soluzione-2).

In [16]:
# Tocca a te: crea risposta_2 e calcola i due prodotti scalari.

### Forme compatibili e più vincoli

`reshape` cambia la forma mantenendo gli elementi. Una dimensione `-1` viene dedotta.
Un vettore `(n,)`, una matrice riga `(1, n)` e una matrice colonna `(n, 1)` sono
oggetti con forme diverse. Per moltiplicare $(m,n)$ per $(n,)$ otteniamo $(m,)$.

`np.ones(n)` crea n valori uguali a 1, `np.zeros(n)` n zeri.
`np.vstack(...)` impila righe: usiamo una riga per il costo e una per il numero di attività.
Qui la seconda riga è solo illustrativa: **non aggiungiamo ancora un vincolo sul numero**.

In [17]:
numero_attivita = len(id_attivita)
riga_costi = costi.reshape(1, -1)
colonna_costi = costi.reshape(-1, 1)
print("Vettore, riga, colonna:", costi.shape, riga_costi.shape, colonna_costi.shape)
matrice_esempio = np.vstack([costi, np.ones(numero_attivita)])
print("Forma di A:", matrice_esempio.shape, "Forma di x:", x_esempio.shape)
print("A @ x (euro, numero di attività):", matrice_esempio @ x_esempio)

Vettore, riga, colonna: (5,) (1, 5) (5, 1)
Forma di A: (2, 5) Forma di x: (5,)
A @ x (euro, numero di attività): [900.   3.]


**Lettura:** `A @ x` restituisce 900 euro e 3 attività. Il calcolo applica un prodotto
scalare a ogni riga. Scrivere `A * x` moltiplicherebbe ogni colonna per la corrispondente
decisione, senza sommare: non è lo stesso risultato.

`sum(axis=0)` somma **lungo il primo asse**, producendo una somma per colonna;
`sum(axis=1)` produce una somma per riga. Usiamo una matrice di conteggi separata,
per non sommare euro e numeri di attività tra loro.

In [18]:
conteggi = np.array([[1, 2, 3], [4, 5, 6]])  # due gruppi, tre giorni
print("Per giorno (colonne):", conteggi.sum(axis=0))
print("Per gruppo (righe):", conteggi.sum(axis=1))
print("Totale:", conteggi.sum())

Per giorno (colonne): [5 7 9]
Per gruppo (righe): [ 6 15]
Totale: 21


### Una copia protegge i dati originali

Uno slicing NumPy può essere una **vista** sugli stessi dati. Se vuoi modificare il
risultato senza cambiare l’array originale, usa `.copy()`. Non modificare mai
`costi` per sperimentare con una variante del problema.

In [19]:
costi_prova = costi[:2].copy()
costi_prova[0] = 0
print("Copia modificata:", costi_prova)
print("Originale:", costi)

Copia modificata: [  0. 300.]
Originale: [400. 300. 500. 200. 600.]


<a id="pandas"></a>
## 3. pandas: tabelle e coefficienti

### DataFrame e Series

Un **DataFrame** è una tabella; una **Series** è una colonna con etichette.
Creiamo la tabella dalle liste già definite e usiamo l’ID come indice con `set_index`.
`columns` elenca le colonne, `dtypes` i loro tipi. Non tutte le colonne devono avere
lo stesso tipo: nomi testuali e costi numerici convivono nella stessa tabella.
`list(...)` converte una sequenza di etichette in una lista Python.

In [20]:
tabella = pd.DataFrame({
    "ID": id_attivita, "Attività": nomi_attivita,
    "Costo": costi_lista, "Beneficio": benefici_lista,
}).set_index("ID")
display(tabella)
print("Colonne:", list(tabella.columns))
print(tabella.dtypes)

Colonne: ['Attività', 'Costo', 'Beneficio']
Attività     object
Costo         int64
Beneficio     int64
dtype: object


,Attività,Costo,Beneficio
ID,,,
A,Torneo sportivo,400,8
B,Laboratorio foto,300,6
C,Rassegna cinema,500,9
D,Scambio libri,200,5
E,Concerto,600,11


`tabella["Costo"]` estrae una Series; `tabella[["Costo", "Beneficio"]]` un DataFrame.
`.loc` seleziona tramite **etichette**, `.iloc` tramite **posizioni**.
In `.loc[righe, colonne]` specifichiamo entrambe le selezioni;
`:` significa tutte le righe o tutte le colonne.

In [21]:
display(tabella["Costo"])
display(tabella[["Costo", "Beneficio"]])
display(tabella.loc[["B", "D"], ["Costo", "Beneficio"]])
display(tabella.iloc[1:3, :])  # posizioni 1 e 2: B e C

ID
A    400
B    300
C    500
D    200
E    600
Name: Costo, dtype: int64

,Costo,Beneficio
ID,,
A,400,8
B,300,6
C,500,9
D,200,5
E,600,11


,Costo,Beneficio
ID,,
B,300,6
D,200,5


,Attività,Costo,Beneficio
ID,,,
B,Laboratorio foto,300,6
C,Rassegna cinema,500,9


Un filtro booleano seleziona righe. Per combinare filtri pandas usiamo `&` (e)
oppure `|` (o), con ogni confronto tra parentesi. Per una sola condizione basta
la maschera. Una nuova colonna nasce con un assegnamento per nome.

In [22]:
display(tabella.loc[tabella["Costo"] <= 400])
analisi = tabella.copy()
analisi["Beneficio per euro"] = analisi["Beneficio"] / analisi["Costo"]
tabella_ordinata = analisi.sort_values("Beneficio per euro", ascending=False, kind="stable")
display(tabella_ordinata)

,Attività,Costo,Beneficio
ID,,,
A,Torneo sportivo,400,8
B,Laboratorio foto,300,6
D,Scambio libri,200,5


,Attività,Costo,Beneficio,Beneficio per euro
ID,,,,
D,Scambio libri,200,5,0.025000
A,Torneo sportivo,400,8,0.020000
B,Laboratorio foto,300,6,0.020000
E,Concerto,600,11,0.018333
C,Rassegna cinema,500,9,0.018000


`ascending=False` richiede ordine decrescente; `kind="stable"` conserva l’ordine
precedente in caso di parità. Il rapporto beneficio/costo è **descrittivo**:
seguire quell’ordine non garantisce l’ottimo binario. Nel nostro esempio, scegliere
prima D e poi A e B dà beneficio 19; più avanti troveremo una scelta da 20.

**Micro-esercizio 3** Da `tabella`, seleziona le attività con costo
al massimo 500 euro **e** beneficio almeno 8. Mostra solo le colonne `Attività`
e `Costo`. [Soluzione](#soluzione-3).

In [23]:
# Tocca a te: costruisci la maschera e seleziona righe e colonne con .loc.

### Il rischio dell’ordinamento

`.to_numpy()` produce un array nell’ordine corrente delle righe; `.to_dict()` conserva
l’associazione tra indice e valore. Dopo un ordinamento, usare i coefficienti come
se fossero ancora in ordine A–E associa i numeri alle decisioni sbagliate.
Ripristiniamo **esplicitamente** l’ordine con `.loc[id_attivita]` prima della conversione.

In [24]:
print("Ordine dopo il sort:", list(tabella_ordinata.index))
costi_disallineati = tabella_ordinata["Costo"].to_numpy()
print("Costo con associazioni SBAGLIATE:", costi_disallineati @ x_esempio)
tabella_modello = tabella_ordinata.loc[id_attivita]
costi_modello = tabella_modello["Costo"].to_numpy(dtype=float)
benefici_modello = tabella_modello["Beneficio"].to_numpy(dtype=float)
costi_per_id = tabella_modello["Costo"].to_dict()
benefici_per_id = tabella_modello["Beneficio"].to_dict()
print("Ordine ripristinato:", list(tabella_modello.index))
print("Costo corretto:", costi_modello @ x_esempio)

Ordine dopo il sort: ['D', 'A', 'B', 'E', 'C']
Costo con associazioni SBAGLIATE: 1200
Ordine ripristinato: ['A', 'B', 'C', 'D', 'E']
Costo corretto: 900.0


**Micro-esercizio 4** Estrai da `tabella_ordinata` la colonna `Costo`
come array, nell’ordine **E, A, D**. Scrivi prima la selezione `.loc`, poi la
conversione. [Soluzione](#soluzione-4).

In [25]:
# Tocca a te: crea risposta_4 senza modificare tabella_ordinata.

### CSV senza file esterni

`to_csv` scrive una tabella in formato CSV. Senza un percorso restituisce una stringa.
`StringIO`, dal modulo standard `io`, permette di leggere quella stringa come un file.
`read_csv(..., index_col="ID")` ripristina l’ID come indice. Non occorre caricare dati
manualmente né collegare Google Drive.

In [26]:
from io import StringIO
testo_csv = tabella.to_csv(index=True)
tabella_riletta = pd.read_csv(StringIO(testo_csv), index_col="ID")
display(tabella_riletta)

,Attività,Costo,Beneficio
ID,,,
A,Torneo sportivo,400,8
B,Laboratorio foto,300,6
C,Rassegna cinema,500,9
D,Scambio libri,200,5
E,Concerto,600,11


**Quale rappresentazione useremo?** Gli array mantengono visibili le dimensioni della
formulazione matriciale SciPy. I dizionari per ID permettono di scrivere variabili e
somme leggibili in Gurobi. NumPy e pandas aiutano a preparare i dati: non sono entrambi
necessari per ogni uso di `gurobipy`.

<a id="scipy"></a>
## 4. SciPy

NumPy fornisce array e operazioni numeriche; SciPy costruisce su questi strumenti
algoritmi di calcolo scientifico.

| Modulo | Esempi di utilizzo |
|---|---|
| `scipy.linalg` | Sistemi lineari e fattorizzazioni |
| `scipy.stats` | Distribuzioni e metodi statistici |
| `scipy.integrate` | Integrali numerici ed equazioni differenziali |
| `scipy.optimize` | Ottimizzazione e ricerca di zeri |

Un solo esempio statistico: per $Z$ normale standard, `norm.cdf(z)` calcola
$P(Z\leq z)$. Sottraendo due valori della funzione di ripartizione otteniamo
$P(-1.96\leq Z\leq 1.96)$.

In [27]:
from scipy.stats import norm
probabilita = norm.cdf(1.96) - norm.cdf(-1.96)
print("Probabilità nell'intervallo:", probabilita)

Probabilità nell'intervallo: 0.950004209703559


### Tre famiglie di problemi

| Problema | Variabili e struttura | Una funzione SciPy |
|---|---|---|
| Continuo non lineare | Variabili reali, funzioni anche non lineari | `minimize` |
| Lineare continuo | Obiettivo e vincoli lineari, variabili reali | `linprog` |
| Lineare intero o misto intero | Obiettivo e vincoli lineari, alcune variabili intere | `milp` |

Useremo soltanto **`milp`**, che chiama il risolutore **HiGHS**.
SciPy è l’interfaccia; HiGHS è il motore di risoluzione. Confronteremo questa modalità
con l’API Python `gurobipy` e il risolutore Gurobi.

<a id="attivita"></a>
## 5. Primo confronto: selezione delle attività

### La formulazione viene prima del codice

- $J=\{A,B,C,D,E\}$ è l’insieme delle attività.
- $a_j$ è il costo in euro e $b_j$ il beneficio in punti dell’attività $j$.
- $K$ è il budget, inizialmente 1.000 euro.
- $x_j=1$ se finanziamo l’attività $j$, $x_j=0$ altrimenti.

$$
\begin{aligned}
\max_x\quad &\sum_{j\in J}b_jx_j\\
\text{con}\quad &\sum_{j\in J}a_jx_j\leq K,\\
&x_j\in\{0,1\},\qquad j\in J.
\end{aligned}
$$

Non si può finanziare metà attività. Cerchiamo un insieme di attività con beneficio
totale massimo, senza superare il budget. Il riferimento matematico per i dati base
è **B, C, D**, costo **1.000**, beneficio **20**: lo verificheremo con il codice.

### SciPy: obiettivo, limiti, integralità, vincoli

`milp` **minimizza** `c @ x`: per massimizzare i benefici scegliamo
`c = -benefici_modello`. Questo `c` dell’API **non è il vettore dei costi**.

`Bounds` imposta $0\leq x\leq1$; `integrality=1` impone valori interi.
**L’integralità da sola non impone il dominio binario.** Servono entrambi.

`LinearConstraint` usa $b_l\leq Ax\leq b_u$. Qui $A$ ha una riga con i costi,
$b_l=-\infty$ (nessun limite inferiore sulla somma) e $b_u=K$.
`np.inf` rappresenta l’infinito. `dtype=int` in `ones` crea indicatori interi.

In [ ]:
obiettivo_scipy = -benefici_modello
limiti_attivita = Bounds(np.zeros(numero_attivita), np.ones(numero_attivita))
integralita_attivita = np.ones(numero_attivita, dtype=int)
matrice_budget = costi_modello.reshape(1, -1)  # una sola riga
disponibilita_budget = np.array([budget], dtype=float)
vincolo_budget = LinearConstraint(matrice_budget, -np.inf, disponibilita_budget)
print("c:", obiettivo_scipy.shape, "A:", matrice_budget.shape)
print("b_u:", disponibilita_budget.shape, "integrality:", integralita_attivita.shape)

c: (5,) A: (1, 5)
b_u: (1,) integrality: (5,)


`milp(...)` restituisce un oggetto risultato. `status` e `message` descrivono l’esito.
`success=True` indica l’ottimalità dichiarata dal solver, entro le sue tolleranze;
non è il solo modo in cui può essere disponibile una soluzione.

| `status` SciPy | Interpretazione |
|---:|---|
| 0 | Ottimo |
| 1 | Limite di iterazioni o tempo: controllare se esiste una soluzione |
| 2 | Inammissibile |
| 3 | Illimitato |
| 4 | Altro esito: leggere il messaggio |

Per leggere il vettore e il valore controlliamo che **sia `x` sia `fun` siano diversi
da `None`**. Una soluzione disponibile dopo un’interruzione non certifica l’ottimalità.

In [29]:
res_attivita = milp(c=obiettivo_scipy, integrality=integralita_attivita,
                    bounds=limiti_attivita, constraints=vincolo_budget)
print("Stato:", res_attivita.status, "success:", res_attivita.success)
print("Messaggio:", res_attivita.message)
sol_scipy_attivita = None
val_scipy_attivita = None
if res_attivita.x is not None and res_attivita.fun is not None:
    sol_scipy_attivita = res_attivita.x.copy()
    val_scipy_attivita = -res_attivita.fun  # torniamo alla massimizzazione
    print("Vettore restituito:", sol_scipy_attivita)
    print("Beneficio:", val_scipy_attivita)

Stato: 0 success: True
Messaggio: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Vettore restituito: [0. 1. 1. 1. 0.]
Beneficio: 20.0


### Gurobi: variabili per ID ed espressioni

Un **generatore** produce i termini uno alla volta: `sum(b[j] for j in scelta)`
somma gli stessi valori di una lista costruita con una comprensione, senza creare
la lista completa. Nella cella seguente le due somme sono numeriche.

In [30]:
termini_beneficio = [benefici_per_id[j] for j in scelta_esempio]
print("Somma della lista:", sum(termini_beneficio))
print("Somma con generatore:", sum(benefici_per_id[j] for j in scelta_esempio))

Somma della lista: 19
Somma con generatore: 19


`gp.Model` crea il modello; `addVars` crea una variabile per ogni ID.
`GRB.BINARY` impone valori 0/1. `gp.quicksum` combina i termini che ora contengono
**variabili del modello**, costruendo un’espressione simbolica.
`setObjective` e `addConstr` traducono obiettivo e vincolo; `optimize` risolve.
`Params.OutputFlag = 0` riduce i messaggi di avanzamento, senza modificare il modello.

In [31]:
modello_attivita = gp.Model("selezione_attivita")
modello_attivita.Params.OutputFlag = 0
x_attivita = modello_attivita.addVars(id_attivita, vtype=GRB.BINARY, name="x")
modello_attivita.setObjective(
    gp.quicksum(benefici_per_id[j] * x_attivita[j] for j in id_attivita), GRB.MAXIMIZE)
modello_attivita.addConstr(
    gp.quicksum(costi_per_id[j] * x_attivita[j] for j in id_attivita) <= budget,
    name="budget")
modello_attivita.optimize()

Restricted license - for non-production use only - expires 2027-11-29


`Status == GRB.OPTIMAL` indica ottimalità entro le tolleranze.
`SolCount > 0` indica una soluzione disponibile. Solo dopo questo controllo leggiamo
gli attributi `.X` delle variabili e `ObjVal` del modello.
Salviamo valori e stato prima di `dispose()`, che libera il modello quando non serve più.

In [32]:
stato_gp_attivita = modello_attivita.Status
print("Status Gurobi:", stato_gp_attivita, "SolCount:", modello_attivita.SolCount)
sol_gp_attivita = None
val_gp_attivita = None
if modello_attivita.SolCount > 0:
    sol_gp_attivita = np.array([x_attivita[j].X for j in id_attivita])
    val_gp_attivita = modello_attivita.ObjVal
    print("Vettore restituito:", sol_gp_attivita, "Beneficio:", val_gp_attivita)
modello_attivita.dispose()

Status Gurobi: 2 SolCount: 2
Vettore restituito: [0. 1. 1. 1. 0.] Beneficio: 20.0


### Quattro controlli, prima di arrotondare

Per una soluzione numerica $\hat x$ controlliamo, con tolleranza assoluta
$\tau=10^{-6}$, limiti, vincoli, integralità e valore obiettivo ricalcolato.
Per esempio, l’integralità richiede
$|\hat x_j-\operatorname{round}(\hat x_j)|\leq\tau$ per ogni componente.

`np.all` richiede che tutte le condizioni di un array siano vere; `np.abs` calcola
valori assoluti; `np.rint` restituisce gli interi più vicini come numeri decimali.
**Nel controllo `rint` serve solo a misurare una distanza: non sostituisce i valori del solver.**
`np.isclose(..., atol=..., rtol=0)` confronta due valori con la sola tolleranza assoluta;
`np.isfinite` esclude infiniti e valori non numerici.

La funzione seguente riunisce questi controlli per i nostri modelli interi con
$0\leq x\leq u$ e $Ax\leq r$. **Non costruisce né risolve modelli.**

In [33]:
tolleranza = 1e-6

def verifica_soluzione(soluzione, valore, profitti, matrice, risorse, massimi):
    finiti = np.all(np.isfinite(soluzione)) and np.isfinite(valore)
    limiti = np.all(soluzione >= -tolleranza) and np.all(soluzione <= massimi + tolleranza)
    vincoli = np.all(matrice @ soluzione <= risorse + tolleranza)
    interi = np.all(np.abs(soluzione - np.rint(soluzione)) <= tolleranza)
    obiettivo = np.isclose(profitti @ soluzione, valore, atol=tolleranza, rtol=0)
    return {"Valori finiti": bool(finiti), "Limiti": bool(limiti),
            "Vincoli": bool(vincoli), "Integralità": bool(interi),
            "Obiettivo coerente": bool(obiettivo),
            "Ammissibile": bool(finiti and limiti and vincoli and interi)}

Ora eseguiamo gli stessi controlli sulle due soluzioni disponibili.
Le terne nel ciclo contengono nome, vettore e valore. `all` è l’analogo Python di
“tutti veri”, qui applicato ai valori del dizionario.
Se una verifica fallisce, `assert` lo segnala: non correggiamo silenziosamente la soluzione.
Solo **dopo** i controlli usiamo `rint(...).astype(int)` per presentare le decisioni come interi.
`pd.DataFrame(esiti_attivita)` raccoglie i dizionari dei controlli in colonne;
`.T` scambia righe e colonne, così ogni riga della tabella rappresenta un solver.

In [34]:
esiti_attivita = {}
for nome, soluzione, valore in [("SciPy/HiGHS", sol_scipy_attivita, val_scipy_attivita),
                               ("Gurobi", sol_gp_attivita, val_gp_attivita)]:
    esiti_attivita[nome] = {"Ammissibile": None}
    if soluzione is not None and valore is not None:
        esiti_attivita[nome] = verifica_soluzione(
            soluzione, valore, benefici_modello, matrice_budget,
            disponibilita_budget, np.ones(numero_attivita))
        assert all(esiti_attivita[nome].values()), "Controllo numerico fallito: attività."
        decisioni = np.rint(soluzione).astype(int)
        print(nome, "scelta:", identificativi[decisioni == 1],
              "costo:", costi_modello @ soluzione, "beneficio:", valore)
display(pd.DataFrame(esiti_attivita).T)  # .T scambia righe e colonne

SciPy/HiGHS scelta: ['B' 'C' 'D'] costo: 1000.0 beneficio: 20.0
Gurobi scelta: ['B' 'C' 'D'] costo: 1000.0 beneficio: 20.0


,Valori finiti,Limiti,Vincoli,Integralità,Obiettivo coerente,Ammissibile
SciPy/HiGHS,True,True,True,True,True,True
Gurobi,True,True,True,True,True,True


**Lettura:** la soluzione B, C, D rispetta il budget e produce 20 punti.
A, B, D era ammissibile ma produceva 19 punti. La verifica di ammissibilità controlla
la scelta; la certificazione di ottimalità viene dal solver, e sarà confermata qui
anche da un’enumerazione indipendente opzionale.

**Micro-esercizio 5** Porta il budget a **1.200 euro** in una
nuova chiamata SciPy, mantenendo invariati costi e benefici. Usa un nuovo oggetto
`LinearConstraint`, senza modificare `budget` o `vincolo_budget`.
Quali attività vengono scelte? La nuova scelta contiene tutte quelle precedenti?
[Soluzione](#soluzione-5).

In [35]:
# Tocca a te: definisci budget_nuovo e un nuovo vincolo. Conserva il problema base.

<a id="produzione"></a>
## 6. Secondo confronto: variabili intere non binarie

Produciamo unità indivisibili di due prodotti. Un’unità del prodotto 1 dà profitto 3,
un’unità del prodotto 2 dà profitto 5. Disponiamo di due risorse, entrambe in quantità 8:
il prodotto 1 ne consuma rispettivamente 2 e 1, il prodotto 2 rispettivamente 1 e 2.
Le unità di ciascuna risorsa sono coerenti all’interno della relativa riga.

$$
\begin{aligned}
\max\quad &3x_1+5x_2\\
\text{con}\quad &2x_1+x_2\leq8,\\
&x_1+2x_2\leq8,\\
&x_1,x_2\in\mathbb Z_{\geq0}.
\end{aligned}
$$

**Intero non significa binario**: possiamo produrre più di un’unità.
Riferimento matematico per i dati base: $(x_1,x_2)=(2,3)$, profitto **21**.
Introduciamo i dati di questo secondo problema in un unico punto.

In [36]:
# DATI BASE DELLA PRODUZIONE: colonne nell'ordine prodotto 1, prodotto 2.
profitti_lista = [3, 5]
consumi_lista = [[2, 1], [1, 2]]
risorse_lista = [8, 8]
profitti = np.array(profitti_lista, dtype=float)
consumi = np.array(consumi_lista, dtype=float)
risorse = np.array(risorse_lista, dtype=float)
print("Profitti:", profitti.shape, "Matrice:", consumi.shape, "Risorse:", risorse.shape)

Profitti: (2,) Matrice: (2, 2) Risorse: (2,)


**SciPy.** Il limite superiore sulle variabili diventa infinito, l’integralità resta 1.
Le due righe della matrice descrivono i due vincoli. Anche qui cambiamo il segno
dei profitti per la minimizzazione e lo ripristiniamo quando leggiamo il risultato.
`np.full(n, valore)` crea un array di lunghezza `n` riempito con quel valore.

In [37]:
limiti_produzione = Bounds(np.zeros(2), np.full(2, np.inf))  # full riempie un array
vincoli_produzione = LinearConstraint(consumi, -np.inf, risorse)
res_produzione = milp(c=-profitti, integrality=np.ones(2, dtype=int),
                      bounds=limiti_produzione, constraints=vincoli_produzione)
print("Stato:", res_produzione.status, "Messaggio:", res_produzione.message)
sol_scipy_produzione = None
val_scipy_produzione = None
if res_produzione.x is not None and res_produzione.fun is not None:
    sol_scipy_produzione = res_produzione.x.copy()
    val_scipy_produzione = -res_produzione.fun

Stato: 0 Messaggio: Optimization terminated successfully. (HiGHS Status 7: Optimal)


**Gurobi.** Usiamo `GRB.INTEGER`, limite inferiore `lb=0` e superiore `ub=GRB.INFINITY`.
Gli indici 0 e 1 corrispondono ai prodotti 1 e 2. I coefficienti provengono dagli
stessi array usati da SciPy; `consumi[riga, colonna]` seleziona un elemento della matrice.

In [38]:
modello_produzione = gp.Model("produzione")
modello_produzione.Params.OutputFlag = 0
x_produzione = modello_produzione.addVars(2, vtype=GRB.INTEGER,
                                          lb=0, ub=GRB.INFINITY, name="x")
modello_produzione.setObjective(
    gp.quicksum(profitti[j] * x_produzione[j] for j in range(2)), GRB.MAXIMIZE)
modello_produzione.addConstr(consumi[0, 0] * x_produzione[0] +
                            consumi[0, 1] * x_produzione[1] <= risorse[0])
modello_produzione.addConstr(consumi[1, 0] * x_produzione[0] +
                            consumi[1, 1] * x_produzione[1] <= risorse[1])
modello_produzione.optimize()

In [39]:
stato_gp_produzione = modello_produzione.Status
print("Status Gurobi:", stato_gp_produzione, "SolCount:", modello_produzione.SolCount)
sol_gp_produzione = None
val_gp_produzione = None
if modello_produzione.SolCount > 0:
    sol_gp_produzione = np.array([x_produzione[j].X for j in range(2)])
    val_gp_produzione = modello_produzione.ObjVal
modello_produzione.dispose()

Status Gurobi: 2 SolCount: 2


Applichiamo i controlli già introdotti. Le risorse utilizzate sono `consumi @ soluzione`:
per (2, 3) ci aspettiamo **7 e 8**, quindi resta una unità della prima risorsa.

In [40]:
esiti_produzione = {}
for nome, soluzione, valore in [("SciPy/HiGHS", sol_scipy_produzione, val_scipy_produzione),
                               ("Gurobi", sol_gp_produzione, val_gp_produzione)]:
    esiti_produzione[nome] = {"Ammissibile": None}
    if soluzione is not None and valore is not None:
        esiti_produzione[nome] = verifica_soluzione(
            soluzione, valore, profitti, consumi, risorse, np.full(2, np.inf))
        assert all(esiti_produzione[nome].values()), "Controllo numerico fallito: produzione."
        print(nome, "quantità:", np.rint(soluzione).astype(int), "profitto:", valore)
        print("Risorse utilizzate:", consumi @ soluzione)
display(pd.DataFrame(esiti_produzione).T)

SciPy/HiGHS quantità: [2 3] profitto: 21.0
Risorse utilizzate: [7. 8.]
Gurobi quantità: [2 3] profitto: 21.0
Risorse utilizzate: [7. 8.]


,Valori finiti,Limiti,Vincoli,Integralità,Obiettivo coerente,Ammissibile
SciPy/HiGHS,True,True,True,True,True,True
Gurobi,True,True,True,True,True,True


### Confronto degli obiettivi e tabella finale

Confrontiamo i valori nella convenzione originale, cioè di **massimizzazione**.
Se entrambi i solver dichiarano l’ottimalità, devono concordare entro tolleranza.
Non richiediamo vettori identici: in altri problemi possono esistere più ottimi.

Per la tabella associamo un testo ai codici tramite dizionari.
`.get(chiave, testo_predefinito)` restituisce il valore della chiave se esiste,
altrimenti il testo predefinito. I codici non riportati richiedono la documentazione.

In [41]:
if res_attivita.success and stato_gp_attivita == GRB.OPTIMAL:
    assert sol_scipy_attivita is not None and sol_gp_attivita is not None
    assert np.isclose(val_scipy_attivita, val_gp_attivita, atol=tolleranza, rtol=0)
if res_produzione.success and stato_gp_produzione == GRB.OPTIMAL:
    assert sol_scipy_produzione is not None and sol_gp_produzione is not None
    assert np.isclose(val_scipy_produzione, val_gp_produzione, atol=tolleranza, rtol=0)

stati_scipy = {0: "Ottimo", 1: "Limite raggiunto", 2: "Inammissibile",
              3: "Illimitato", 4: "Altro: leggere message"}
stati_gurobi = {GRB.OPTIMAL: "Ottimo", GRB.INFEASIBLE: "Inammissibile",
               GRB.TIME_LIMIT: "Limite di tempo"}

In [42]:
riepilogo = pd.DataFrame([
    ["Attività", "SciPy/HiGHS", stati_scipy.get(res_attivita.status, "Altro stato"),
     sol_scipy_attivita is not None, val_scipy_attivita, esiti_attivita["SciPy/HiGHS"]["Ammissibile"]],
    ["Attività", "Gurobi", stati_gurobi.get(stato_gp_attivita, "Altro stato"),
     sol_gp_attivita is not None, val_gp_attivita, esiti_attivita["Gurobi"]["Ammissibile"]],
    ["Produzione", "SciPy/HiGHS", stati_scipy.get(res_produzione.status, "Altro stato"),
     sol_scipy_produzione is not None, val_scipy_produzione, esiti_produzione["SciPy/HiGHS"]["Ammissibile"]],
    ["Produzione", "Gurobi", stati_gurobi.get(stato_gp_produzione, "Altro stato"),
     sol_gp_produzione is not None, val_gp_produzione, esiti_produzione["Gurobi"]["Ammissibile"]],
], columns=["Problema", "Risolutore", "Stato", "Soluzione disponibile", "Obiettivo", "Ammissibile"])
display(riepilogo)

,Problema,Risolutore,Stato,Soluzione disponibile,Obiettivo,Ammissibile
0,Attività,SciPy/HiGHS,Ottimo,True,20.0,True
1,Attività,Gurobi,Ottimo,True,20.0,True
2,Produzione,SciPy/HiGHS,Ottimo,True,21.0,True
3,Produzione,Gurobi,Ottimo,True,21.0,True


| Aspetto | SciPy `milp` | `gurobipy` |
|---|---|---|
| Rappresentazione | Vettori, matrice dei vincoli, limiti | Variabili indicizzate ed espressioni |
| Obiettivo | Minimizzazione; cambiamo segno per massimizzare | Senso esplicito, per esempio `GRB.MAXIMIZE` |
| Identità delle variabili | L’ordine delle colonne va mantenuto | Gli ID rendono leggibili le espressioni |
| Risultato | Oggetto `res`, con stato e possibili `x`, `fun` | Attributi del modello e delle variabili |

Le due formulazioni rappresentano lo stesso problema matematico. Queste istanze non
costituiscono un benchmark e non permettono conclusioni generali sulla velocità.

**Micro-esercizio 6** Porta la disponibilità della prima risorsa
a **10**, mantenendo la seconda a 8. Crea una copia di `risorse`, poi un nuovo modello
Gurobi. Controlla quantità, profitto e consumi. [Soluzione](#soluzione-6).

In [43]:
# Tocca a te: usa risorse.copy() e un nuovo modello, senza modificare i dati base.

<a id="riepilogo"></a>
## 7. Riepilogo

Nei prossimi laboratori riutilizzeremo:

- **Liste, dizionari e cicli** per organizzare insiemi e dati.
- **Array, forme e prodotti matriciali** per leggere costi, obiettivi e vincoli.
- **DataFrame, filtri e ID** per preparare coefficienti senza perdere le associazioni.
- **Variabili, obiettivo e vincoli** per tradurre un problema in codice Gurobi.
- **Stato, disponibilità e controlli numerici** per interpretare correttamente il risultato.

Domanda conclusiva: *se il solver termina per un limite di tempo senza restituire una
soluzione, possiamo dichiarare il modello inammissibile?* **No**: l’assenza di una
soluzione trovata non equivale a una prova di inammissibilità.

<a id="opzionale"></a>
## 8. Opzionale: perché non basta arrotondare?

Nel **rilassamento continuo** della produzione permettiamo quantità reali non negative.
Usiamo ancora `milp`, ma con `integrality=0` per ogni componente.
L’ottimo atteso è $(8/3,8/3)$, con valore $64/3$, un limite superiore all’ottimo intero.

`np.floor` arrotonda ogni componente per difetto. Confronteremo la soluzione continua,
l’arrotondamento al più vicino e quello per difetto. È un esperimento esplicito:
gli arrotondamenti **non vengono passati per soluzioni del solver**.

In [44]:
res_continuo = milp(c=-profitti, integrality=np.zeros(2, dtype=int),
                    bounds=limiti_produzione, constraints=vincoli_produzione)
print("Stato:", res_continuo.status, "Messaggio:", res_continuo.message)
if res_continuo.x is not None and res_continuo.fun is not None:
    continuo = res_continuo.x.copy()
    assert np.all(np.isfinite(continuo)) and np.isfinite(res_continuo.fun)
    assert np.all(continuo >= -tolleranza)
    assert np.all(consumi @ continuo <= risorse + tolleranza)
    assert np.isclose(profitti @ continuo, -res_continuo.fun, atol=tolleranza, rtol=0)
    vicino = np.rint(continuo)
    difetto = np.floor(continuo)
    for nome, proposta in [("Continuo", continuo), ("Più vicino", vicino), ("Per difetto", difetto)]:
        print(nome, proposta, "profitto:", profitti @ proposta,
              "vincoli rispettati:", np.all(consumi @ proposta <= risorse + tolleranza))

Stato: 0 Messaggio: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Continuo [2.66666667 2.66666667] profitto: 21.333333333333332 vincoli rispettati: True
Più vicino [3. 3.] profitto: 24.0 vincoli rispettati: False
Per difetto [2. 2.] profitto: 16.0 vincoli rispettati: True


Per i dati base, $(3,3)$ usa 9 unità di ogni risorsa: **non è ammissibile**.
$(2,2)$ è ammissibile ma vale 16, meno di 21. Quindi, anche quando produce una scelta
ammissibile, l’arrotondamento può perdere l’ottimo. In questo esempio arrotondare per
difetto preserva i vincoli grazie ai coefficienti non negativi e ai soli limiti superiori;
non è una proprietà universale dei problemi interi.

<a id="soluzioni"></a>
## 9. Soluzioni commentate dei micro-esercizi

<a id="soluzione-1"></a>
### Esercizio 1 · filtro con un ciclo
Manteniamo gli ID nell’ordine iniziale e aggiungiamo solo quelli che soddisfano la soglia.

In [45]:
soluzione_1 = []
for identificativo in id_attivita:
    if costi_per_id[identificativo] <= 400:
        soluzione_1.append(identificativo)
print(soluzione_1)

['A', 'B', 'D']


<a id="soluzione-2"></a>
### Esercizio 2 · prodotto scalare
B e D occupano le posizioni 1 e 3, contando da zero.

In [46]:
soluzione_2 = np.array([0, 1, 0, 1, 0])
print("Costo:", costi @ soluzione_2, "Beneficio:", benefici @ soluzione_2)
print("Budget rispettato:", costi @ soluzione_2 <= budget)

Costo: 500.0 Beneficio: 11.0
Budget rispettato: True


<a id="soluzione-3"></a>
### Esercizio 3 · filtro su due colonne
Le parentesi separano i confronti; `&` richiede che siano entrambi veri per ciascuna riga.

In [47]:
maschera_3 = (tabella["Costo"] <= 500) & (tabella["Beneficio"] >= 8)
soluzione_3 = tabella.loc[maschera_3, ["Attività", "Costo"]]
display(soluzione_3)

,Attività,Costo
ID,,
A,Torneo sportivo,400
C,Rassegna cinema,500


<a id="soluzione-4"></a>
### Esercizio 4 · ordine esplicito degli ID
`.loc` segue l’ordine delle etichette richieste, indipendentemente dal precedente ordinamento.

In [48]:
soluzione_4 = tabella_ordinata.loc[["E", "A", "D"], "Costo"].to_numpy()
print(soluzione_4)

[600 400 200]


<a id="soluzione-5"></a>
### Esercizio 5 · un budget diverso
Cambiamo soltanto il lato destro del vincolo. Restano invariati obiettivo, domini e coefficienti.

In [49]:
budget_nuovo = 1200
vincolo_nuovo = LinearConstraint(matrice_budget, -np.inf, [budget_nuovo])
res_budget_nuovo = milp(c=-benefici_modello, integrality=integralita_attivita,
                        bounds=limiti_attivita, constraints=vincolo_nuovo)
print("Stato:", res_budget_nuovo.status, "Messaggio:", res_budget_nuovo.message)
if res_budget_nuovo.x is not None and res_budget_nuovo.fun is not None:
    controlli_5 = verifica_soluzione(res_budget_nuovo.x, -res_budget_nuovo.fun,
        benefici_modello, matrice_budget, np.array([budget_nuovo]), np.ones(numero_attivita))
    assert all(controlli_5.values())
    scelta_5 = np.rint(res_budget_nuovo.x).astype(int)
    print("Scelta:", identificativi[scelta_5 == 1],
          "Costo:", costi_modello @ res_budget_nuovo.x, "Beneficio:", -res_budget_nuovo.fun)

Stato: 0 Messaggio: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Scelta: ['A' 'D' 'E'] Costo: 1200.0 Beneficio: 24.0


Con i dati base, A, D, E costano 1.200 euro e valgono 24. B e C escono dalla scelta:
più budget non implica conservare tutte le attività prima selezionate.

<a id="soluzione-6"></a>
### Esercizio 6 · una risorsa più abbondante
La copia evita di cambiare i vincoli del problema base. Ricostruiamo esplicitamente il modello.

In [50]:
risorse_nuove = risorse.copy()
risorse_nuove[0] = 10
modello_nuovo = gp.Model("produzione_variante")
modello_nuovo.Params.OutputFlag = 0
x_nuove = modello_nuovo.addVars(2, vtype=GRB.INTEGER, lb=0, ub=GRB.INFINITY)
modello_nuovo.setObjective(gp.quicksum(profitti[j] * x_nuove[j] for j in range(2)), GRB.MAXIMIZE)
modello_nuovo.addConstr(consumi[0, 0] * x_nuove[0] + consumi[0, 1] * x_nuove[1] <= risorse_nuove[0])
modello_nuovo.addConstr(consumi[1, 0] * x_nuove[0] + consumi[1, 1] * x_nuove[1] <= risorse_nuove[1])
modello_nuovo.optimize()
print("Status Gurobi:", modello_nuovo.Status, "SolCount:", modello_nuovo.SolCount)
if modello_nuovo.SolCount > 0:
    quantita_6 = np.array([x_nuove[j].X for j in range(2)])
    controlli_6 = verifica_soluzione(quantita_6, modello_nuovo.ObjVal,
                                    profitti, consumi, risorse_nuove, np.full(2, np.inf))
    assert all(controlli_6.values())
    print("Quantità:", np.rint(quantita_6).astype(int), "Profitto:", modello_nuovo.ObjVal)
    print("Risorse utilizzate:", consumi @ quantita_6)
modello_nuovo.dispose()

Status Gurobi: 2 SolCount: 2
Quantità: [4 2] Profitto: 22.0
Risorse utilizzate: [10.  8.]


Con la prima risorsa a 10, le quantità (4, 2) utilizzano (10, 8) e producono profitto 22.
La disponibilità aggiuntiva cambia sia la scelta sia il valore ottimo.

<a id="enumerazione"></a>
## 10. Opzionale: verifica indipendente degli ottimi

Per le cinque attività esistono solo $2^5=32$ scelte binarie. `itertools.product`
genera tutte le combinazioni, senza chiamare alcun solver. Usiamo le **liste originali**,
non la matrice costruita per SciPy. `elif` aggiunge un secondo caso al blocco `if`.

Per ogni scelta ammissibile ricalcoliamo costo e beneficio e conserviamo tutti gli ottimi.
È una verifica adatta a queste istanze minuscole: il numero di combinazioni cresce
esponenzialmente con il numero di attività.

In [51]:
from itertools import product
miglior_beneficio = -np.inf
ottimi_attivita = []
for scelta in product([0, 1], repeat=len(id_attivita)):
    costo = sum(costi_lista[j] * scelta[j] for j in range(len(id_attivita)))
    beneficio = sum(benefici_lista[j] * scelta[j] for j in range(len(id_attivita)))
    if costo <= budget:
        if beneficio > miglior_beneficio:
            miglior_beneficio = beneficio
            ottimi_attivita = [scelta]
        elif beneficio == miglior_beneficio:
            ottimi_attivita.append(scelta)
print("Enumerazione attività:", ottimi_attivita, "Valore:", miglior_beneficio)

Enumerazione attività: [(0, 1, 1, 1, 0)] Valore: 20


Per la produzione, tutti i consumi sono positivi. Ogni risorsa fornisce un limite
superiore sulla quantità di ciascun prodotto: ne prendiamo il minimo e la parte intera.
I cicli comprendono gli estremi superiori grazie a `+ 1`.

In [52]:
limite_1 = int(min(risorse_lista[k] / consumi_lista[k][0] for k in range(2)))
limite_2 = int(min(risorse_lista[k] / consumi_lista[k][1] for k in range(2)))
miglior_profitto = -np.inf
ottimi_produzione = []
for q1 in range(limite_1 + 1):
    for q2 in range(limite_2 + 1):
        uso_1 = consumi_lista[0][0] * q1 + consumi_lista[0][1] * q2
        uso_2 = consumi_lista[1][0] * q1 + consumi_lista[1][1] * q2
        if uso_1 <= risorse_lista[0] and uso_2 <= risorse_lista[1]:
            profitto = profitti_lista[0] * q1 + profitti_lista[1] * q2
            if profitto > miglior_profitto:
                miglior_profitto = profitto
                ottimi_produzione = [(q1, q2)]
            elif profitto == miglior_profitto:
                ottimi_produzione.append((q1, q2))
print("Enumerazione produzione:", ottimi_produzione, "Valore:", miglior_profitto)

Enumerazione produzione: [(2, 3)] Valore: 21


Confrontiamo l’enumerazione con ciascun risultato dichiarato ottimo, senza richiedere
che i vettori coincidano. Gli ultimi controlli verificano i valori di riferimento dei
**dati base forniti**: se cambi la cella dei dati, dovrai aggiornare questi riferimenti.

In [53]:
if res_attivita.success:
    assert np.isclose(val_scipy_attivita, miglior_beneficio, atol=tolleranza, rtol=0)
if stato_gp_attivita == GRB.OPTIMAL:
    assert np.isclose(val_gp_attivita, miglior_beneficio, atol=tolleranza, rtol=0)
if res_produzione.success:
    assert np.isclose(val_scipy_produzione, miglior_profitto, atol=tolleranza, rtol=0)
if stato_gp_produzione == GRB.OPTIMAL:
    assert np.isclose(val_gp_produzione, miglior_profitto, atol=tolleranza, rtol=0)
assert miglior_beneficio == 20 and ottimi_attivita == [(0, 1, 1, 1, 0)]
assert miglior_profitto == 21 and ottimi_produzione == [(2, 3)]
print("I due ottimi di riferimento sono confermati per enumerazione.")

I due ottimi di riferimento sono confermati per enumerazione.


## Riferimenti

- [Python: tutorial](https://docs.python.org/3/tutorial/)
- [NumPy: guida introduttiva](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [pandas: tutorial iniziali](https://pandas.pydata.org/docs/getting_started/intro_tutorials/)
- [SciPy: milp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html)
- [SciPy: normale standard](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [Gurobi: API Python](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/overview.html)
- [Gurobi: installazione e licenze in Colab](https://support.gurobi.com/hc/en-us/articles/4409582394769-Google-Colab-Installation-and-Licensing)
- [Gurobi: codici di stato](https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html)
- [Colab: FAQ](https://research.google.com/colaboratory/faq.html)